# 02 · Part D 2024: filter to GLP-1 drugs with DuckDB, then profile

**Goal:** turn the 4.4 GB CMS *Medicare Part D Prescribers by Provider and Drug, 2024* file (28 million rows) into a small, clean GLP-1 subset that every later notebook can load in a second.

**Input:** `data/01_raw/medicare_part_d_prescribers/medicare_part_d_prescribers_by_provider_and_drug_2024.csv` (never edited)
**Output:** `data/02_interim/part_d_2024_glp1.parquet`

### What one row means
One row = **one prescriber × one drug × one year**. For example: "Dr X in Ohio wrote 120 Mounjaro claims for 25 Medicare patients in 2024, costing US$150k."
- `Tot_Clms` = number of prescriptions **including refills**
- `Tot_Benes` = number of distinct patients (beneficiaries)
- `Tot_Drug_Cst` = total amount paid (US$), before rebates

### Why DuckDB and not pandas? (interview question)
| | pandas `read_csv` | DuckDB |
|---|---|---|
| Memory | Loads **everything** into RAM. 4.4 GB of text is roughly 10–20 GB as Python objects, which crashes a laptop | **Streams** the file in chunks, so memory stays small |
| Storage model | Row by row | **Columnar + vectorised**: reads only the columns a query needs and processes about 2,000 values at a time with fast CPU instructions |
| Language | Python methods | **SQL**: the language analysts use everywhere (Snowflake, BigQuery, Databricks) |
| Setup | — | No server: an *in-process* database, like SQLite but built for analytics (OLAP) |

**The rule of thumb:** use a query engine (DuckDB, Spark, a warehouse) to **reduce** the data first, then use pandas on the small result.

In [1]:
from pathlib import Path
import time
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.1f}".format)

# Find the project root (the folder that contains data/), whether run from notebooks/ or the root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "01_raw").exists())
RAW_CSV = ROOT / "data/01_raw/medicare_part_d_prescribers/medicare_part_d_prescribers_by_provider_and_drug_2024.csv"
OUT_PARQUET = ROOT / "data/02_interim/part_d_2024_glp1.parquet"

print("CSV size:", round(RAW_CSV.stat().st_size / 1e9, 2), "GB")
con = duckdb.connect()  # in-memory database; nothing is saved unless we write it out
con.sql("SET enable_progress_bar = false")

CSV size: 4.43 GB


## Step 1 · Look before you load: the schema

`DESCRIBE` makes DuckDB *sniff* the CSV: it reads a sample, guesses each column's type, and returns instantly without scanning 4.4 GB.

**A data-type trap (good interview story):** some columns *look* numeric but are really **identifiers**:
- `Prscrbr_NPI` is a 10-digit doctor ID. Nobody adds NPIs together, so we store it as text.
- `Prscrbr_State_FIPS` is a state code like `01` (Alabama). Stored as a number, the leading zero disappears and later joins break.

So we override those two columns to `VARCHAR` (text) instead of trusting the guess.

In [2]:
CSV_TYPES = {"Prscrbr_NPI": "VARCHAR", "Prscrbr_State_FIPS": "VARCHAR"}
src = f"read_csv('{RAW_CSV.as_posix()}', header=true, types={CSV_TYPES})"

con.sql(f"DESCRIBE SELECT * FROM {src}").df()[["column_name", "column_type"]]

,column_name,column_type
0,Prscrbr_NPI,VARCHAR
1,Prscrbr_Last_Org_Name,VARCHAR
2,Prscrbr_First_Name,VARCHAR
3,Prscrbr_City,VARCHAR
4,Prscrbr_State_Abrvtn,VARCHAR
5,Prscrbr_State_FIPS,VARCHAR
6,Prscrbr_Type,VARCHAR
7,Prscrbr_Type_Src,VARCHAR
8,Brnd_Name,VARCHAR
9,Gnrc_Name,VARCHAR


In [3]:
# Names are public data, but we show doctors aggregated by default, so hide them in previews
con.sql(f"SELECT * EXCLUDE (Prscrbr_Last_Org_Name, Prscrbr_First_Name) FROM {src} LIMIT 5").df()

,Prscrbr_NPI,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_Type,Prscrbr_Type_Src,Brnd_Name,Gnrc_Name,Tot_Clms,Tot_30day_Fills,Tot_Day_Suply,Tot_Drug_Cst,Tot_Benes,GE65_Sprsn_Flag,GE65_Tot_Clms,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Tot_Day_Suply,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes
0,1003000126,Bethesda,MD,24,Internal Medicine,Claim-Specialty,Abrysvo,"Rsv Vacc, Pref A And Pref B/Pf",20,20.0,20,"5,994.0",20,NaN,20,20.0,"5,994.0",20,NaN,20
1,1003000126,Bethesda,MD,24,Internal Medicine,Claim-Specialty,Boostrix Tdap,"Diphth,pertuss(Acell),tet Vac",12,12.0,12,648.3,12,NaN,12,12.0,648.3,12,NaN,12
2,1003000142,Toledo,OH,39,Anesthesiology,Claim-Specialty,Acetaminophen-Codeine,Acetaminophen With Codeine,24,24.0,628,362.5,11,#,<NA>,NaN,NaN,<NA>,*,<NA>
3,1003000142,Toledo,OH,39,Anesthesiology,Claim-Specialty,Amitriptyline Hcl,Amitriptyline Hcl,47,47.0,1410,271.2,16,NaN,30,30.0,150.2,900,*,<NA>
4,1003000142,Toledo,OH,39,Anesthesiology,Claim-Specialty,Baclofen,Baclofen,47,47.0,1395,333.4,26,NaN,33,33.0,222.1,975,#,<NA>


## Step 2 · Count all rows (one full scan)

This query reads the whole file once. Watch the time: DuckDB uses all your CPU cores in parallel.
We expect **28,023,892** rows (the number recorded when the file was downloaded, so this doubles as a check that the file is complete).

In [4]:
t = time.time()
n_rows = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
print(f"{n_rows:,} rows in {time.time() - t:.0f}s")
assert n_rows == 28_023_892, "Row count differs from the verified download — file may be incomplete"

28,023,892 rows in 8s


## Step 3 · Which drug names count as GLP-1? Discover, don't guess

Before filtering, we list every generic name that looks like a GLP-1 receptor agonist (names ending in *-glutide* or *-enatide*, plus tirzepatide).
That way the filter is based on what is **actually in the data**, not on memory, and we can see edge cases (combination products, spelling variants).

In [5]:
con.sql(f'''
    SELECT Gnrc_Name, Brnd_Name, COUNT(*) AS n_rows, SUM(Tot_Clms) AS claims
    FROM {src}
    WHERE regexp_matches(lower(Gnrc_Name), 'glutide|enatide|tirzepatide')
    GROUP BY ALL
    ORDER BY claims DESC
''').df()

,Gnrc_Name,Brnd_Name,n_rows,claims
0,Semaglutide,Ozempic,171191,"9,930,484.0"
1,Tirzepatide,Mounjaro,97208,"4,694,896.0"
2,Dulaglutide,Trulicity,101479,"3,816,376.0"
3,Semaglutide,Rybelsus,35474,"899,704.0"
4,Insulin Glargine/Lixisenatide,Soliqua 100-33,6568,"189,307.0"
5,Exenatide Microspheres,Bydureon Bcise,4844,"85,420.0"
6,Semaglutide,Wegovy,3956,"76,176.0"
7,Liraglutide,Victoza 3-Pak,2765,"44,376.0"
8,Insulin Degludec/Liraglutide,Xultophy 100-3.6,1110,"28,517.0"
9,Liraglutide,Victoza 2-Pak,1417,"27,346.0"


### Filter decision
We keep the seven GLP-1 generic names below (incl. `Insulin Degludec/Liraglutide` = Xultophy, an insulin + GLP-1 combination).

**A trap the discovery step caught:** Bydureon BCise is listed as `Exenatide Microspheres`, not `Exenatide`. An exact match on "exenatide" alone silently drops 4,844 rows. Always look at the real values before you filter.

**Excluded on purpose:**
- `Teduglutide` (Gattex) matched the pattern but is a **GLP-2** drug for short bowel syndrome, a false positive of the regex.
- `Insulin Glargine/Lixisenatide` (Soliqua) is a basal-insulin-first combination, small in volume and not part of the tirzepatide-vs-semaglutide story. Write such choices down: in an interview, *"why did you include X but not Y?"* is a common question.

**Why `lower(trim(...))`?** Text in public files often has inconsistent case or stray spaces. Normalising both sides makes the match robust.

In [6]:
GLP1_GENERICS = [
    "tirzepatide",                    # Mounjaro, Zepbound (Lilly)
    "semaglutide",                    # Ozempic, Wegovy, Rybelsus (Novo Nordisk)
    "dulaglutide",                    # Trulicity (Lilly)
    "liraglutide",                    # Victoza, Saxenda, generics
    "exenatide",                      # Byetta
    "exenatide microspheres",         # Bydureon BCise (CMS uses a different generic name!)
    "insulin degludec/liraglutide",   # Xultophy
]
generic_list_sql = ", ".join(f"'{g}'" for g in GLP1_GENERICS)

## Step 4 · Filter and write Parquet (second full scan)

`COPY (query) TO 'file.parquet'` streams the matching rows straight to disk, so pandas never sees the 28M rows.

**Why Parquet instead of CSV?** (another interview favourite)
- **Columnar**: reading 2 of 22 columns only touches those 2 columns on disk
- **Compressed** (we use ZSTD): typically 5–10× smaller than CSV
- **Typed**: the schema (text vs number) is stored inside the file, so no re-guessing types and no lost leading zeros
- Read natively by pandas, DuckDB, Spark, Polars and Hugging Face `datasets`

We also add a cleaned `Brand` column (trimmed) and sort by NPI so rows for the same doctor sit together, which helps compression and later joins.

In [7]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
t = time.time()
con.sql(f'''
    COPY (
        SELECT *,
               trim(Brnd_Name) AS Brand,
               trim(Gnrc_Name) AS Generic
        FROM {src}
        WHERE lower(trim(Gnrc_Name)) IN ({generic_list_sql})
        ORDER BY Prscrbr_NPI, Brand
    ) TO '{OUT_PARQUET.as_posix()}' (FORMAT parquet, COMPRESSION zstd)
''')
print(f"Wrote {OUT_PARQUET.relative_to(ROOT)} in {time.time() - t:.0f}s, "
      f"{OUT_PARQUET.stat().st_size / 1e6:.1f} MB (vs {RAW_CSV.stat().st_size / 1e9:.2f} GB CSV)")

Wrote data\02_interim\part_d_2024_glp1.parquet in 17s, 11.4 MB (vs 4.43 GB CSV)


## Step 5 · Validate the output

From here on we only read the small Parquet file. Each query below runs in milliseconds.
Expected (recorded on 18 Oct 2026): **420,364 rows** and **199,831 distinct prescribers**.

In [8]:
glp1 = f"read_parquet('{OUT_PARQUET.as_posix()}')"
check = con.sql(f'''
    SELECT COUNT(*)                     AS rows,
           COUNT(DISTINCT Prscrbr_NPI)  AS prescribers,
           COUNT(DISTINCT Brand)        AS brands,
           SUM(Tot_Clms)                AS claims,
           SUM(Tot_Drug_Cst) / 1e9      AS cost_usd_bn
    FROM {glp1}
''').df()
assert (check.loc[0, "rows"], check.loc[0, "prescribers"]) == (420_364, 199_831), "Subset differs from verified numbers"
check

,rows,prescribers,brands,claims,cost_usd_bn
0,420364,199831,12,"19,620,006.0",24.6


### Data-quality checks
1. **Uniqueness:** one row per (prescriber, brand, generic). If this fails, later joins would double-count.
2. **Missing values:** CMS **suppresses** small counts to protect patient privacy. `Tot_Benes` is blank when fewer than 11 patients, and the `GE65_*` columns carry flags (`*` = suppressed because fewer than 11; `#` = suppressed so the small number can't be worked out by subtraction). So blanks here are **by design, not errors**, and must never be filled with 0.
3. **Sanity:** no negative claims or costs.

In [9]:
dupes = con.sql(f'''
    SELECT COUNT(*) FROM (
        SELECT Prscrbr_NPI, Brand, Generic FROM {glp1} GROUP BY ALL HAVING COUNT(*) > 1)
''').fetchone()[0]
negatives = con.sql(f"SELECT COUNT(*) FROM {glp1} WHERE Tot_Clms < 0 OR Tot_Drug_Cst < 0").fetchone()[0]
print("Duplicate (NPI, brand, generic) keys:", dupes)
print("Rows with negative claims/cost:   ", negatives)

null_share = con.sql(f'''
    SELECT ROUND(100 * AVG(CASE WHEN Tot_Benes      IS NULL THEN 1 ELSE 0 END), 1) AS pct_Tot_Benes_suppressed,
           ROUND(100 * AVG(CASE WHEN GE65_Tot_Benes IS NULL THEN 1 ELSE 0 END), 1) AS pct_GE65_Benes_missing,
           ROUND(100 * AVG(CASE WHEN Prscrbr_Type   IS NULL THEN 1 ELSE 0 END), 1) AS pct_specialty_missing
    FROM {glp1}
''').df()
null_share

Duplicate (NPI, brand, generic) keys: 0
Rows with negative claims/cost:    0


,pct_Tot_Benes_suppressed,pct_GE65_Benes_missing,pct_specialty_missing
0,71.3,94.9,0.0


In [10]:
con.sql(f'''
    SELECT GE65_Sprsn_Flag, GE65_Bene_Sprsn_Flag, COUNT(*) AS rows
    FROM {glp1} GROUP BY ALL ORDER BY rows DESC
''').df()

,GE65_Sprsn_Flag,GE65_Bene_Sprsn_Flag,rows
0,NaN,*,176095
1,#,*,94794
2,*,*,54321
3,NaN,#,48337
4,#,#,25198
5,NaN,NaN,21619


## Step 6 · First profile: the 2024 Medicare GLP-1 market

### By brand
Note: `prescribers` counts doctors who wrote **at least 11 claims** for that drug. CMS drops rows with fewer than 11 claims entirely, so small prescribers are invisible. That is a limitation to state in the README.

In [11]:
by_brand = con.sql(f'''
    SELECT Brand, Generic,
           SUM(Tot_Clms)                         AS claims,
           SUM(Tot_Drug_Cst) / 1e6               AS cost_usd_m,
           COUNT(DISTINCT Prscrbr_NPI)           AS prescribers,
           SUM(Tot_Drug_Cst) / SUM(Tot_30day_Fills) AS cost_per_30day_fill
    FROM {glp1}
    GROUP BY ALL
    ORDER BY claims DESC
''').df()
by_brand["claims_share_pct"] = 100 * by_brand["claims"] / by_brand["claims"].sum()
by_brand

,Brand,Generic,claims,cost_usd_m,prescribers,cost_per_30day_fill,claims_share_pct
0,Ozempic,Semaglutide,"9,930,484.0","12,376.0",171191,904.6,50.6
1,Mounjaro,Tirzepatide,"4,694,896.0","5,825.3",97208,"1,039.5",23.9
2,Trulicity,Dulaglutide,"3,816,376.0","4,786.8",101479,946.9,19.5
3,Rybelsus,Semaglutide,"899,704.0","1,309.3",35474,916.2,4.6
4,Bydureon Bcise,Exenatide Microspheres,"85,420.0",89.3,4844,797.1,0.4
5,Wegovy,Semaglutide,"76,176.0",107.3,3956,"1,328.5",0.4
6,Victoza 3-Pak,Liraglutide,"44,376.0",44.7,2765,722.3,0.2
7,Xultophy 100-3.6,Insulin Degludec/Liraglutide,"28,517.0",42.9,1110,937.4,0.1
8,Victoza 2-Pak,Liraglutide,"27,346.0",17.6,1417,583.8,0.1
9,Liraglutide,Liraglutide,"14,832.0",9.7,771,586.2,0.1


**Story hook:** Zepbound (tirzepatide **for obesity**) is almost zero in Medicare 2024, because Medicare did not pay for weight-loss drugs before the **Medicare GLP-1 Bridge (July 2026)**. The same molecule sold for diabetes (Mounjaro) is large. So the doctors already prescribing Mounjaro and Ozempic to older patients are a natural starting list for obesity targeting in 2026. Notebook 05 turns this into a score.

### By manufacturer (molecule-level)

In [12]:
con.sql(f'''
    SELECT CASE WHEN Generic IN ('Tirzepatide', 'Dulaglutide') THEN 'Eli Lilly'
                WHEN Generic IN ('Semaglutide', 'Liraglutide', 'Insulin Degludec/Liraglutide') THEN 'Novo Nordisk*'
                ELSE 'Other (exenatide: AstraZeneca/Amylin)' END AS manufacturer,
           SUM(Tot_Clms) AS claims,
           ROUND(100 * SUM(Tot_Clms) / SUM(SUM(Tot_Clms)) OVER (), 1) AS claims_share_pct,
           SUM(Tot_Drug_Cst) / 1e6 AS cost_usd_m
    FROM {glp1} GROUP BY ALL ORDER BY claims DESC
''').df()

,manufacturer,claims,claims_share_pct,cost_usd_m
0,Novo Nordisk*,"11,021,435.0",56.2,"13,907.5"
1,Eli Lilly,"8,511,286.0",43.4,"10,612.1"
2,Other (exenatide: AstraZeneca/Amylin),"87,285.0",0.4,90.9


\* Generic liraglutide (from other manufacturers) is grouped with Novo here for simplicity; its volume is small.

### Who prescribes GLP-1s? Top specialties
`Prscrbr_Type` is the doctor's specialty as reported to CMS. This tells sales teams **which kinds of doctors** to focus on, and it is a key column in the doctor segmentation (notebook 05).

In [13]:
con.sql(f'''
    SELECT Prscrbr_Type AS specialty,
           COUNT(DISTINCT Prscrbr_NPI) AS prescribers,
           SUM(Tot_Clms)              AS claims,
           ROUND(100 * SUM(Tot_Clms) / SUM(SUM(Tot_Clms)) OVER (), 1) AS claims_share_pct,
           ROUND(SUM(Tot_Clms) / COUNT(DISTINCT Prscrbr_NPI), 0)     AS claims_per_prescriber
    FROM {glp1} GROUP BY ALL ORDER BY claims DESC LIMIT 15
''').df()

,specialty,prescribers,claims,claims_share_pct,claims_per_prescriber
0,Family Practice,64038,"5,794,176.0",29.5,90.0
1,Internal Medicine,44984,"4,890,255.0",24.9,109.0
2,Nurse Practitioner,51791,"4,050,021.0",20.6,78.0
3,Endocrinology,5881,"2,504,082.0",12.8,426.0
4,Physician Assistant,17518,"1,376,160.0",7.0,79.0
5,General Practice,2385,"190,263.0",1.0,80.0
6,Cardiology,2309,"165,279.0",0.8,72.0
7,Pharmacist,1022,"116,413.0",0.6,114.0
8,Nephrology,1148,"83,642.0",0.4,73.0
9,Geriatric Medicine,795,"54,687.0",0.3,69.0


### Tirzepatide vs semaglutide share by specialty
The key competitive question: **where is Lilly's tirzepatide strong or weak against Novo's semaglutide?**

In [14]:
con.sql(f'''
    WITH s AS (
        SELECT Prscrbr_Type AS specialty,
               SUM(Tot_Clms) FILTER (WHERE Generic = 'Tirzepatide') AS tirzepatide,
               SUM(Tot_Clms) FILTER (WHERE Generic = 'Semaglutide') AS semaglutide,
               SUM(Tot_Clms) AS all_glp1
        FROM {glp1} GROUP BY ALL
    )
    SELECT *, ROUND(100 * tirzepatide / (tirzepatide + semaglutide), 1) AS tirz_share_of_tirz_plus_sema_pct
    FROM s WHERE all_glp1 > 100000 ORDER BY all_glp1 DESC
''').df()

,specialty,tirzepatide,semaglutide,all_glp1,tirz_share_of_tirz_plus_sema_pct
0,Family Practice,"1,299,946.0","3,293,689.0","5,794,176.0",28.3
1,Internal Medicine,"1,117,261.0","2,779,475.0","4,890,255.0",28.7
2,Nurse Practitioner,"1,038,260.0","2,191,633.0","4,050,021.0",32.1
3,Endocrinology,"694,945.0","1,293,016.0","2,504,082.0",35.0
4,Physician Assistant,"333,034.0","759,480.0","1,376,160.0",30.5
5,General Practice,"36,509.0","102,778.0","190,263.0",26.2
6,Cardiology,"42,684.0","110,507.0","165,279.0",27.9
7,Pharmacist,"16,266.0","71,307.0","116,413.0",18.6


### Geography: top states

In [15]:
con.sql(f'''
    SELECT Prscrbr_State_Abrvtn AS state,
           COUNT(DISTINCT Prscrbr_NPI) AS prescribers,
           SUM(Tot_Clms) AS claims,
           ROUND(100 * SUM(Tot_Clms) FILTER (WHERE Generic = 'Tirzepatide')
                     / SUM(Tot_Clms) FILTER (WHERE Generic IN ('Tirzepatide', 'Semaglutide')), 1) AS tirz_share_pct
    FROM {glp1} GROUP BY ALL ORDER BY claims DESC LIMIT 15
''').df()

,state,prescribers,claims,tirz_share_pct
0,TX,14689,"1,659,229.0",32.5
1,CA,19330,"1,647,966.0",18.9
2,NY,12442,"1,327,074.0",26.5
3,FL,13230,"1,306,286.0",32.8
4,PA,8738,"925,711.0",25.9
5,OH,7743,"766,117.0",30.1
6,NC,7339,"756,282.0",27.9
7,GA,5625,"696,815.0",35.0
8,MI,7105,"689,268.0",38.7
9,IL,6802,"622,137.0",26.3


### Concentration: do a few doctors write most prescriptions?
Sales teams **decile** doctors: rank them by volume and cut them into 10 equal-sized groups. If the top deciles write most of the claims, a small sales force can cover most of the market. This is the classic *Pareto (80/20)* idea and the first step of targeting.

In [16]:
con.sql(f'''
    WITH doc AS (SELECT Prscrbr_NPI, SUM(Tot_Clms) AS claims FROM {glp1} GROUP BY 1),
         ranked AS (SELECT *, NTILE(10) OVER (ORDER BY claims DESC) AS decile FROM doc)
    SELECT decile,
           COUNT(*) AS prescribers,
           SUM(claims) AS claims,
           ROUND(100 * SUM(claims) / SUM(SUM(claims)) OVER (), 1) AS claims_share_pct,
           ROUND(100 * SUM(SUM(claims)) OVER (ORDER BY decile) / SUM(SUM(claims)) OVER (), 1) AS cumulative_pct
    FROM ranked GROUP BY decile ORDER BY decile
''').df()

,decile,prescribers,claims,claims_share_pct,cumulative_pct
0,1,19984,"8,125,461.0",41.4,41.4
1,2,19983,"3,415,381.0",17.4,58.8
2,3,19983,"2,340,193.0",11.9,70.7
3,4,19983,"1,722,503.0",8.8,79.5
4,5,19983,"1,289,020.0",6.6,86.1
5,6,19983,"958,230.0",4.9,91.0
6,7,19983,"701,372.0",3.6,94.6
7,8,19983,"495,371.0",2.5,97.1
8,9,19983,"331,423.0",1.7,98.8
9,10,19983,"241,052.0",1.2,100.0


## Summary & what's next

**Done:** 28M rows → GLP-1 subset in `data/02_interim/part_d_2024_glp1.parquet`, validated (row count, uniqueness, suppression understood) and profiled (brand, manufacturer, specialty, state, concentration).

**Limitations to carry forward**
1. Medicare Part D only (mostly patients aged 65+). No commercial insurance, cash-pay or Medicaid at doctor level.
2. Rows with fewer than 11 claims are dropped by CMS, so low-volume prescribers are invisible.
3. `Tot_Drug_Cst` is the gross amount before manufacturer rebates, not company revenue.
4. Obesity use (Zepbound, Wegovy) is almost absent because it wasn't covered in 2024. The 2026 opportunity must be **modelled**, not observed.

**Next:** Notebook 01 fetches NPPES, Open Payments, FAERS, labels, PubMed and ClinicalTrials.gov. Then notebook 03 joins everything on `Prscrbr_NPI` into the one-row-per-doctor **HCP 360** table.

### Interview questions this notebook prepares you for
- *Why DuckDB over pandas for a 4 GB file?* Streaming, columnar, vectorised, parallel; reduce first, then use pandas.
- *Why Parquet?* Columnar, compressed, typed, works across tools.
- *Why store NPI/FIPS as text?* They are identifiers, and leading zeros matter.
- *What does a blank `Tot_Benes` mean?* Privacy suppression (fewer than 11 patients), not zero; never impute it as 0.
- *What does deciling show?* How concentrated prescribing is, which decides sales-force focus.